# Baseline EnerGNN — `LocalSumMessagePassingFunction` (référence amont)

Notebook de référence (Approche 0) pour l'équipe R&D RTE. Il déroule le pipeline EnerGNN d'origine (référence amont) sur le serveur GPU de La Javaness : la `LocalSumMessagePassingFunction` (somme par couple `(classe, port)`) intégrée dans un `RecurrentCoupler`, entraînée sur deux substrats.

**Prérequis :**
- Le dépôt cloné dans `./energnn-attention/` sur le serveur GPU de La Javaness (ou un chemin équivalent).
- `uv sync --extra gpu` exécuté.
- `pypowsybl==1.15.0` et `pypowsybl-to-energnn` (mode editable) installés.
- Branche `main` active.

**Durée attendue :** 5 à 10 minutes au total sur le serveur GPU de La Javaness.

**Structure en deux parties :**
- **Partie 1** — expériences sur LinearSystem (toy DC power flow).
- **Partie 2** — expériences sur ieee9 et ieee14 (AC load flow réel).

## 1. Mise en place de l'environnement

Imports : JAX, NumPy, Flax NNX, optax. Vérification du device JAX et du dtype par défaut. On attend `CudaDevice` (serveur GPU de La Javaness) et `float32`.


In [1]:
import sys
from pathlib import Path

import jax
import jax.numpy as jnp
import numpy as np
import optax
from flax import nnx

# Locate repo root so the notebook works from any cwd.
_root = Path.cwd().resolve()
while not (_root / "pyproject.toml").exists() and _root.parent != _root:
    _root = _root.parent
sys.path.insert(0, str(_root / "benchmarks"))

print(f"JAX version:    {jax.__version__}")
print(f"JAX devices:    {jax.devices()}")
print(f"Default dtype:  {jnp.array(0.0).dtype}")
print(f"Repo root:      {_root}")


JAX version:    0.9.0


JAX devices:    [CudaDevice(id=0), CudaDevice(id=1)]


Default dtype:  float32
Repo root:      /mnt/nasbkp01/GPUDATA/van.tuan/energnn_attention_git


## 2. Méthode — `LocalSumMessagePassingFunction`

EnerGNN représente le réseau électrique sous forme de H2MG (Hyper Heterogeneous Multi Graph) : chaque classe d'hyper-arête (charge, générateur, ligne, etc.) possède plusieurs ports, chaque port pointant vers une adresse. La `LocalSumMessagePassingFunction` est la message function d'origine d'EnerGNN, de formulation :

$$\psi_\theta(h, x)_a = \sigma\left( \sum_{(c, e, o) \in \mathcal{N}_x(a)} \xi^{c, o}_\theta(h_e, x_e) \right)$$

où :
- $\xi^{c, o}_\theta$ est un MLP par couple `(classe, port)` — c'est l'hétérogénéité du H2MG.
- $h_e = (h_{o(e)})_{o \in \mathcal{O}^c}$ est la concaténation des coordonnées aux ports de l'hyper-arête $e$.
- $\sigma$ est l'`outer_activation` appliquée après agrégation.
- $\mathcal{N}_x(a)$ regroupe tous les triplets `(classe, arête, port)` dont le port $o$ de l'arête $e$ pointe vers l'adresse $a$.

L'implémentation s'appuie sur `gather` et `scatter_add` (`energnn.model.utils`) avec `mode='drop'` pour les indices fictifs, masqués via `non_fictitious` avant l'agrégation. L'équivariance par permutation est préservée car `scatter_add` est commutatif sur l'axe source.

C'est la baseline à laquelle toute variante d'attention (notebooks `01_gatv2.ipynb` à `06_virtual_address.ipynb`) doit se comparer.

# Partie 1 — Expériences sur LinearSystem

LinearSystem est le toy problem fourni par EnerGNN : DC power flow sur des graphes aléatoires creux à $n \in \{2, 3\}$ bus. L'objectif de cette partie est de valider que le pipeline LocalSum + `RecurrentCoupler` tourne sur le substrat le plus simple, que la loss décroît de façon monotone et qu'il n'y a pas de NaN. Ceci correspond à la vérification Gate 3 (toy experiment).

## 3.1. Construction de `LinearSystemProblemLoader`

Le loader produit des batches de DC power flow toy : chaque instance est un graphe aléatoire creux à 2 ou 3 bus, l'oracle étant l'angle de phase $\theta^\star$ obtenu en résolvant $\mathbf{B}\theta = \mathbf{P}$. La configuration reprend celle du setup Gate 3 documenté dans `BASELINES.md` (`dataset_size=64`, `batch_size=4`, seeds `[0, 1, 2]`).


In [2]:
from energnn.problem.example import LinearSystemProblemLoader

ls_train = LinearSystemProblemLoader(seed=0)
ls_val = LinearSystemProblemLoader(seed=42)

print(f"train loader: {type(ls_train).__name__}, seed=0")
print(f"val loader:   {type(ls_val).__name__}, seed=42")
print(f"context_structure classes: {list(ls_train.context_structure.hyper_edge_sets.keys())}")
print(f"decision_structure classes: {list(ls_train.decision_structure.hyper_edge_sets.keys())}")


train loader: LinearSystemProblemLoader, seed=0
val loader:   LinearSystemProblemLoader, seed=42
context_structure classes: ['line', 'bus']
decision_structure classes: ['bus']


## 3.2. Inspection d'une instance LinearSystem

Premier batch : on affiche la structure du context graph (nombre d'adresses, classes d'hyper-arêtes, ports, dimensions des features). LinearSystem est compact (3 à 4 classes) comparé à ieee14 (11 classes) — c'est un bon point d'entrée pour se familiariser avec le H2MG.


In [3]:
ls_batch = next(iter(ls_train))
ls_ctx_batch, _ = ls_batch.get_context()
ls_ctx = jax.tree.map(lambda x: x[0], ls_ctx_batch)
n_addr_ls = int(ls_ctx.non_fictitious_addresses.shape[0])

print(f"n_addr (incl padding):  {n_addr_ls}")
print(f"non_fictitious count:   {int(ls_ctx.non_fictitious_addresses.sum())}")
print(f"\nHyper-edge sets:")
for key, hes in ls_ctx.hyper_edge_sets.items():
    n_ports = len(hes.port_dict)
    n_edges = next(iter(hes.port_dict.values())).shape[0] if n_ports > 0 else 0
    feat_dim = hes.feature_array.shape[-1] if hes.feature_array is not None else 0
    print(f"  {key:<22s} n_edges={n_edges:>3d}  n_ports={n_ports}  feat_dim={feat_dim}")


n_addr (incl padding):  4
non_fictitious count:   2

Hyper-edge sets:
  bus                    n_edges=  4  n_ports=1  feat_dim=1
  line                   n_edges=  6  n_ports=2  feat_dim=1


## 3.3. Construction du GNN LocalSum pour LinearSystem

Pipeline à quatre étages (Normalizer → Encoder → Coupler → Decoder), aligné sur la configuration Tiny de `model/ready_to_use.py` :

- `TDigestNormalizer` — normalisation par feature à partir des quantiles t-digest.
- `MLPEncoder` — encode les features brutes vers `latent_dim`.
- `RecurrentCoupler` — englobe la `LocalSumMessagePassingFunction` et l'itère `n_steps` fois.
- `MLPEquivariantDecoder` — décode le latent vers la `decision_structure`.

`encoded_feature_size=LATENT_DIM` informe la message function de la taille des features post-encoder.


In [4]:
from energnn.model import (
    GNN, MLP, MLPEncoder, MLPEquivariantDecoder, RecurrentCoupler,
    LocalSumMessagePassingFunction, TDigestNormalizer,
)
from energnn.trainer import Trainer

LATENT_DIM = 4  # Tiny config

def build_localsum_gnn(in_struct, out_struct, rngs):
    return GNN(
        normalizer=TDigestNormalizer(in_structure=in_struct, n_breakpoints=10, update_limit=1000),
        encoder=MLPEncoder(
            in_structure=in_struct, hidden_sizes=[], activation=nnx.leaky_relu,
            out_size=LATENT_DIM, use_bias=True, final_activation=None, rngs=rngs,
        ),
        coupler=RecurrentCoupler(
            phi=MLP(
                in_size=LATENT_DIM, hidden_sizes=[], activation=nnx.leaky_relu,
                out_size=LATENT_DIM, use_bias=True, final_activation=nnx.tanh, rngs=rngs,
            ),
            message_functions=[
                LocalSumMessagePassingFunction(
                    in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
                    activation=nnx.leaky_relu, out_size=LATENT_DIM, use_bias=True,
                    final_activation=None, outer_activation=nnx.tanh,
                    encoded_feature_size=LATENT_DIM, rngs=rngs,
                )
            ],
            n_steps=4,
        ),
        decoder=MLPEquivariantDecoder(
            in_graph_structure=in_struct, in_array_size=LATENT_DIM, hidden_sizes=[],
            activation=nnx.leaky_relu, out_structure=out_struct, use_bias=True,
            final_activation=None, encoded_feature_size=LATENT_DIM, rngs=rngs,
        ),
    )

ls_gnn = build_localsum_gnn(ls_train.context_structure, ls_train.decision_structure, nnx.Rngs(0))

# Count params via nnx.split
_, params, _ = nnx.split(ls_gnn, nnx.Param, ...)
n_params = sum(int(leaf.size) for leaf in jax.tree_util.tree_leaves(params) if hasattr(leaf, "size"))
print(f"LinearSystem LocalSum Tiny GNN: latent_dim={LATENT_DIM}, n_steps=4, n_params={n_params}")


LinearSystem LocalSum Tiny GNN: latent_dim=4, n_steps=4, n_params=185


## 3.4. Entraînement sur 5 epochs et courbe de loss

Entraînement de la configuration Tiny pendant 5 epochs avec `optax.adam(1e-3)`, évaluation après chaque epoch sur le loader de validation. On vérifie que la loss décroît de façon monotone et qu'aucun NaN n'apparaît. Le run Gate 3 complet (10 epochs, 3 seeds) est documenté dans `BASELINES.md`, chapitre 1.


In [5]:
trainer_ls = Trainer(model=ls_gnn, gradient_transformation=optax.adam(1e-3))
eval_before, _ = trainer_ls.eval(ls_val, progress_bar=False)
print(f"eval before training: {float(eval_before):.4e}")

ls_loss_curve = [float(eval_before)]
for ep in range(5):
    trainer_ls.train(train_loader=ls_train, val_loader=None, n_epochs=1,
                     progress_bar=False, eval_before_training=False, eval_after_epoch=False)
    s, _ = trainer_ls.eval(ls_val, progress_bar=False)
    ls_loss_curve.append(float(s))
    print(f"  epoch {ep+1}: eval = {float(s):.4e}")

assert ls_loss_curve[-1] < ls_loss_curve[0], "loss did not decrease"
print(f"\nfinal/initial ratio: {ls_loss_curve[-1] / ls_loss_curve[0]:.3f}")
print("PASS: loss decreases monotonically over 5 epochs")


eval before training: 1.3402e+00


  epoch 1: eval = 1.2934e+00


  epoch 2: eval = 1.2528e+00


  epoch 3: eval = 1.2128e+00


  epoch 4: eval = 1.1724e+00


  epoch 5: eval = 1.1323e+00

final/initial ratio: 0.845
PASS: loss decreases monotonically over 5 epochs


# Partie 2 — Expériences sur IEEE-9 et IEEE-14 (AC load flow)

Ce substrat mesure la capacité du modèle : AC load flow non linéaire, 11 classes d'hyper-arêtes, oracle calculé par `pypowsybl.lf.run_ac()` pour V_mag, P, Q et I. Il s'agit du référentiel Gate 5. Le notebook se contente d'un run court (3 à 5 epochs) pour rester rapide — les chiffres Gate 5 complets (3 seeds × Tiny/Small × 15 epochs) figurent dans `BASELINES.md`, chapitre 3.

## 4.1. Construction de `ACLoadFlowProblemLoader` (pre-cache actif)

Le loader pré-génère 32 instances une fois lors de `__init__` avec un RNG privé seedé depuis `seed`, puis met en cache les paires `(context, oracle)` sous forme de `JaxGraph` (commit `7d5f9f8`). Les itérations suivantes parcourent ce cache sans rappeler `pypowsybl.lf.run_ac()`. Le résultat est bit-identique au comportement on-the-fly.

On construit deux loaders, l'un pour ieee14, l'autre pour ieee9. Chaque construction prend 5 à 10 secondes (pypowsybl exécute le solveur AC 32 fois).


In [6]:
import time
from ac_load_flow_problem import ACLoadFlowProblemLoader

LOADERS = {}
for net in ("ieee9", "ieee14"):
    t0 = time.perf_counter()
    LOADERS[net] = {
        "train": ACLoadFlowProblemLoader(network_name=net, dataset_size=32, batch_size=4,
                                          seed=0, perturbation_scale=0.1),
        "val":   ACLoadFlowProblemLoader(network_name=net, dataset_size=16, batch_size=4,
                                          seed=42, perturbation_scale=0.1),
    }
    print(f"{net}: built train+val loaders in {time.perf_counter() - t0:.1f}s")


ieee9: built train+val loaders in 7.4s


ieee14: built train+val loaders in 7.3s


## 4.2. Inspection du context graph ieee14

ieee14 compte 14 bus, 17 lignes, 5 générateurs, 11 charges et 1 shunt. Comparé à LinearSystem (3–4 classes, quelques arêtes), ce cas exerce réellement le pipeline sur un H2MG hétérogène dont les feature dims varient (3 à 11 selon la classe).


In [7]:
ieee14_loader = LOADERS["ieee14"]["train"]
ctx14, _ = ieee14_loader._cached_pairs[0]
n_addr14 = int(ctx14.non_fictitious_addresses.shape[0])

print(f"ieee14 n_addr:           {n_addr14}")
print(f"non_fictitious count:    {int(ctx14.non_fictitious_addresses.sum())}")
print(f"\nHyper-edge sets (non-empty):")
for key, hes in ctx14.hyper_edge_sets.items():
    n_ports = len(hes.port_dict)
    n_edges = next(iter(hes.port_dict.values())).shape[0] if n_ports > 0 else 0
    if n_edges == 0:
        continue
    feat_dim = hes.feature_array.shape[-1] if hes.feature_array is not None else 0
    print(f"  {key:<22s} n_edges={n_edges:>3d}  n_ports={n_ports}  feat_dim={feat_dim}")


ieee14 n_addr:           14


non_fictitious count:    14

Hyper-edge sets (non-empty):
  buses                  n_edges= 14  n_ports=1  feat_dim=0
  generators             n_edges=  5  n_ports=2  feat_dim=10
  lines                  n_edges= 17  n_ports=2  feat_dim=8
  loads                  n_edges= 11  n_ports=1  feat_dim=3
  shunts                 n_edges=  1  n_ports=1  feat_dim=8


## 4.3. Entraînement de LocalSum sur ieee9 et ieee14 (Tiny, 3 epochs)

Entraînement de la configuration Tiny (latent_dim 4, n_steps 4) sur les deux réseaux pendant 3 epochs. C'est un run court de vérification du pipeline ; les chiffres Gate 5 complets (Tiny/Small × 10/15 epochs × 3 seeds) sont déjà stockés dans `benchmarks/results/00_baseline/baseline_ac_load_flow.json`.

Attendu :
- une eval qui décroît régulièrement sur les deux réseaux ;
- ieee9 (graphe plus petit) converge plus vite que ieee14.


In [8]:
N_EPOCHS_IEEE = 3
ieee_results = {}

for net in ("ieee9", "ieee14"):
    train_loader = LOADERS[net]["train"]
    val_loader = LOADERS[net]["val"]
    gnn = build_localsum_gnn(train_loader.context_structure, train_loader.decision_structure, nnx.Rngs(0))
    trainer = Trainer(model=gnn, gradient_transformation=optax.adam(1e-3))

    eval_before, _ = trainer.eval(val_loader, progress_bar=False)
    curve = [float(eval_before)]
    t0 = time.perf_counter()
    for ep in range(N_EPOCHS_IEEE):
        trainer.train(train_loader=train_loader, val_loader=None, n_epochs=1,
                      progress_bar=False, eval_before_training=False, eval_after_epoch=False)
        s, _ = trainer.eval(val_loader, progress_bar=False)
        curve.append(float(s))
    train_time = time.perf_counter() - t0
    ieee_results[net] = {"curve": curve, "train_time_s": train_time}
    print(f"{net} Tiny ({N_EPOCHS_IEEE} epochs, {train_time:.1f}s):")
    for ep, v in enumerate(curve):
        label = "init" if ep == 0 else f"ep {ep}"
        print(f"  {label:<6s} eval = {v:.4e}")
    assert curve[-1] < curve[0], f"{net} loss did not decrease"
print("\nPASS: both networks converge under LocalSum Tiny")


ieee9 Tiny (3 epochs, 71.8s):
  init   eval = 6.0039e-01
  ep 1   eval = 5.4504e-01
  ep 2   eval = 4.9655e-01
  ep 3   eval = 4.4985e-01


ieee14 Tiny (3 epochs, 93.1s):
  init   eval = 3.7157e-01
  ep 1   eval = 3.2947e-01
  ep 2   eval = 2.9554e-01
  ep 3   eval = 2.6508e-01

PASS: both networks converge under LocalSum Tiny


## 4.4. Gate 6 — mesure de performance sur ieee14

Mesure du temps médian forward et forward + backward de la `LocalSumMessagePassingFunction` isolée (sans coupler) sur le context ieee14. C'est la référence de coût à laquelle les variantes d'attention se comparent. Le Gate 6 complet (ieee118 + ieee300) est dans `benchmarks/results/perf_gatv2_vs_localsum.json`.


In [9]:
import statistics

mf_perf = LocalSumMessagePassingFunction(
    in_graph_structure=ieee14_loader.context_structure, in_array_size=4, out_size=4,
    hidden_sizes=[4], activation=nnx.leaky_relu, final_activation=None,
    outer_activation=nnx.tanh, seed=64,
)
perf_coords = jnp.full((n_addr14, 4), 0.5, dtype=jnp.float32)

@nnx.jit
def fwd(m, c):
    out, _ = m(graph=ctx14, coordinates=c, get_info=False)
    return out

@nnx.jit
def fwd_bwd(m, c):
    def loss(mod):
        out, _ = mod(graph=ctx14, coordinates=c, get_info=False)
        return jnp.mean(out ** 2)
    return nnx.value_and_grad(loss)(m)

def time_calls(callable_, n_warmup=20, n_measure=100):
    for _ in range(n_warmup):
        out = callable_(); jax.block_until_ready(out)
    timings = []
    for _ in range(n_measure):
        t0 = time.perf_counter()
        out = callable_(); jax.block_until_ready(out)
        timings.append(time.perf_counter() - t0)
    return statistics.median(timings) * 1000.0

fwd_ms = time_calls(lambda: fwd(mf_perf, perf_coords))
fb_ms = time_calls(lambda: fwd_bwd(mf_perf, perf_coords))
print(f"LocalSum on ieee14 (median of 100 calls, JIT):")
print(f"  forward:           {fwd_ms:.3f} ms")
print(f"  forward+backward:  {fb_ms:.3f} ms")


LocalSum on ieee14 (median of 100 calls, JIT):
  forward:           9.298 ms
  forward+backward:  12.756 ms


## 4.5. Gate 7 — déterminisme numérique du forward

Test de propriété pour Gate 7 (reproductibilité bit-identique) : pour un même triplet `(mf, graph, coords)`, plusieurs appels doivent produire des sorties à l'ULP près. Le Gate 7 officiel impose une empreinte bit-à-bit identique (`benchmarks/consistency_gatv2.py`), valable uniquement en exécution depuis un script autonome. Dans Jupyter, vérifier `atol < 1e-6` suffit.

In [10]:
out_a = fwd(mf_perf, perf_coords); jax.block_until_ready(out_a)
out_b = fwd(mf_perf, perf_coords); jax.block_until_ready(out_b)
max_diff = float(jnp.max(jnp.abs(out_a - out_b)))
print(f"max |out_a - out_b|:  {max_diff:.2e}")
assert max_diff < 1e-6, f"forward not numerically deterministic: {max_diff}"
print("PASS: forward numerically deterministic (atol < 1e-6)")


max |out_a - out_b|:  0.00e+00
PASS: forward numerically deterministic (atol < 1e-6)


## 5. Références

Ce notebook fixe la baseline LocalSum sur les deux substrats (LinearSystem + ieee9/14). Toute variante d'attention (`01_gatv2.ipynb` à `06_virtual_address.ipynb`) s'y compare directement.

**Documents associés :**
- `BASELINES.md` — chiffres Gate 5/6/7 complets (3 seeds × Tiny/Small × ieee9/14, perf sur ieee118/300).
- `tests/model/unit/test_message_fonction.py` — tests unitaires de référence pour LocalSum ; chaque variante doit fournir l'équivalent.